# **GAN-based AI Text Detector**

In [4]:
import os, numpy as np, pandas as pd, random, string
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

# ------------------
# Device
# ------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_PATH  = "train_essays.csv"
TEST_PATH   = "test_essays.csv"
PROMPT_PATH = "train_prompts.csv"


# ------------------
# Load data
# ------------------
src_train  = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH) if os.path.exists(PROMPT_PATH) else None
src_sub    = pd.read_csv(TEST_PATH)

# ------------------
# Model prep (tokenizer + BERT backbone)
# ------------------
tokenizer_save_path = "./tok"
model_save_path     = "./chkpt"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
pretrained_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(device)
pretrained_model.eval()  # we'll use it only for embeddings

# We'll take embeddings from the BERT encoder:
embedding_model = pretrained_model.bert  # returns last_hidden_state when called

# ------------------
# Hyperparameters
# ------------------
train_batch_size = 16
test_batch_size  = 64
lr               = 2e-4
beta1            = 0.5
nz               = 100     # latent dim for generator input
num_epochs       = 3
num_hidden_layers= 6       # depth for the small BertEncoder blocks
train_ratio      = 0.8

# ------------------
# Dataset
# ------------------
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, generated):
        self.texts  = list(texts)
        self.generated = list(generated) if generated is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.generated is None:
            return self.texts[idx]
        return self.texts[idx], self.generated[idx]

# Split train into train/test (holdout) for AUC monitoring
all_num   = len(src_train)
train_num = int(all_num * train_ratio)
test_num  = all_num - train_num

train_set = src_train.sample(frac=train_ratio, random_state=42)
val_set   = src_train.drop(train_set.index).reset_index(drop=True)

# Build a single "test_set" dataframe consistent with your scaffold
test_set = pd.concat([
    val_set
]).reset_index(drop=True)

# Build datasets
train_dataset = GANDAIGDataset(train_set["text"], train_set["generated"])
test_dataset  = GANDAIGDataset(test_set["text"],  test_set["generated"])

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size,  shuffle=False)

# ------------------
# Generator
#   - maps noise (B, nz) -> pseudo BERT-like embeddings (B, seq_len, hidden)
#   - then runs a small BertEncoder block to add contextualization
# ------------------
# use pretrained config and override only what we need
config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_hidden_layers=num_hidden_layers,
    output_hidden_states=False,
    output_attentions=False
)

SEQ_LEN = 128  # fixed sequence length for fake embeddings

class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # project noise -> (B, SEQ_LEN*hidden)
        self.fc = nn.Linear(input_dim, SEQ_LEN * config.hidden_size)

        # a light nonlinearity stack
        self.conv_net = nn.Sequential(
            nn.Dropout(0.1),
            nn.GELU(),
        )
        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        # x: (B, nz)
        x = self.fc(x)                                     # (B, SEQ_LEN*H)
        x = x.view(-1, SEQ_LEN, config.hidden_size)        # (B, L, H)
        x = self.conv_net(x)                               # simple smoothing
        # BertEncoder expects hidden_states with shape (B, L, H)
        out = self.bert_encoder(
            hidden_states=x,
            attention_mask=None,
            head_mask=None,
            output_attentions=False,
            output_hidden_states=False,
            return_dict=True
        )
        return out  # return a ModelOutput with .last_hidden_state

# ------------------
# Discriminator (BERT-ish encoder + mean-pool + MLP head)
# ------------------
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: (B, L, H)
        sum_hidden = hidden_states.sum(dim=1)             # (B, H)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)         # (B, 1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask           # (B, H) normalized
        return mean_embeddings

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        # optionally initialize first 6 layers from pretrained encoder
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:num_hidden_layers]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input_embeddings):  # input_embeddings: (B, L, H)
        out = self.bert_encoder(
            hidden_states=input_embeddings,
            attention_mask=None,
            head_mask=None,
            output_attentions=False,
            output_hidden_states=False,
            return_dict=True
        )
        pooled = self.pooler(out.last_hidden_state)  # (B, H)
        logits = self.classifier(pooled)             # (B, 1)
        return torch.sigmoid(logits).view(-1)        # (B,)

# ------------------
# Utilities
# ------------------
def preparation_embedding(texts):
    # texts: list[str]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=SEQ_LEN, return_tensors="pt")
    input_ids     = enc["input_ids"].to(device)
    token_type_ids= enc.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    attention_mask= enc["attention_mask"].to(device)
    with torch.no_grad():
        out = embedding_model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
    # return last_hidden_state (B, L, H)
    return out.last_hidden_state

def eval_auc(model):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            texts, labels = batch[0], batch[1]
            embeded = preparation_embedding(texts)             # (B, L, H)
            embeded = embeded.to(device)
            outputs = model(embeded)                           # (B,)
            predictions.extend(outputs.detach().cpu().numpy())
            actuals.extend(labels.detach().cpu().numpy())
    auc = roc_auc_score(actuals, predictions) if len(set(actuals)) > 1 else 0.5
    print("AUC:", round(auc, 4))
    return auc

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_hidden_layers=num_hidden_layers,
    output_hidden_states=False,
    output_attentions=False,
)
config._attn_implementation = "eager"
if not hasattr(config, "position_embedding_type"):
    config.position_embedding_type = "absolute"

In [6]:

# ------------------
# GAN step
# ------------------
def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.train(); netG.train()
    batch_size = real_data.size(0)

    # 1) Train D on real
    netD.zero_grad()
    real_label = torch.ones(batch_size, device=device)
    out_real = netD(real_data)
    errD_real = criterion(out_real, real_label)
    errD_real.backward()
    D_x = out_real.mean().item()

    # 2) Train D on fake
    noise = torch.randn(batch_size, nz, device=device)
    fake_out = netG(noise).last_hidden_state             # (B, L, H)
    fake_label = torch.zeros(batch_size, device=device)
    out_fake = netD(fake_out.detach())
    errD_fake = criterion(out_fake, fake_label)
    errD_fake.backward()
    D_G_z1 = out_fake.mean().item()
    optimizerD.step()

    # 3) Train G to fool D
    netG.zero_grad()
    target_real_for_G = torch.ones(batch_size, device=device)
    out_fake_for_G = netD(fake_out)
    errG = criterion(out_fake_for_G, target_real_for_G)
    errG.backward()
    D_G_z2 = out_fake_for_G.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print(f'[{epoch+1}/{num_epochs}][{i}/{len(train_loader)}] '
              f'Loss_D: {(errD_real+errD_fake).item():.4f} '
              f'Loss_G: {errG.item():.4f} D(x): {D_x:.4f} '
              f'D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}')
    return optimizerG, optimizerD, netG, netD

# ------------------
# Build models & opts
# ------------------
netG = Generator(nz).to(device)
netD = Discriminator().to(device)

criterion  = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# ------------------
# Train
# ------------------
model_infos = []
for epoch in range(num_epochs):
    for i, batch in enumerate(train_loader, 0):
        texts, labels = batch[0], batch[1]
        with torch.no_grad():
            embeded = preparation_embedding(texts)  # (B, L, H)
        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=labels.float().to(device),  # unused (GAN labels are generated inside)
            epoch=epoch, i=i
        )
    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete!')

# ------------------
# Inference (use best AUC Discriminator)
# ------------------
max_auc_model_info = max(model_infos, key=lambda d: d['auc_score'])
model = Discriminator().to(device)
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.eval()

class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts): self.texts = list(texts)
    def __getitem__(self, idx): return self.texts[idx]
    def __len__(self): return len(self.texts)

# Build test dataset/loader
sub_dataset = InferenceDataset(src_sub["text"])
inference_loader = DataLoader(sub_dataset, batch_size=test_batch_size, shuffle=False)

sub_predictions = []
with torch.no_grad():
    for batch in inference_loader:
        texts = batch
        embeded = preparation_embedding(texts).to(device)
        outputs = model(embeded)
        sub_predictions.extend(outputs.detach().cpu().numpy())

# Submission DF
if "id" in src_sub.columns:
    sub_ans_df = pd.DataFrame({"id": src_sub["id"], "prediction": sub_predictions})
else:
    sub_ans_df = pd.DataFrame({"prediction": sub_predictions})
print(sub_ans_df.head())

[1/3][0/68] Loss_D: 100.0000 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 1.0000 / 1.0000
[1/3][50/68] Loss_D: 93.7500 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 0.9375 / 1.0000
AUC: 0.5
[2/3][0/68] Loss_D: 100.0000 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 1.0000 / 1.0000
[2/3][50/68] Loss_D: 93.7500 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 0.9375 / 1.0000
AUC: 0.5
[3/3][0/68] Loss_D: 100.0000 Loss_G: 0.0000 D(x): 1.0000 D(G(z)): 1.0000 / 1.0000
[3/3][50/68] Loss_D: 112.5000 Loss_G: 0.0000 D(x): 0.8750 D(G(z)): 1.0000 / 1.0000
AUC: 0.5
Train complete!
         id  prediction
0  0000aaaa         1.0
1  1111bbbb         1.0
2  2222cccc         1.0
